In [1]:
import numpy as np
import pandas as pd

# -----------------------------
# Parámetros generales
# -----------------------------
np.random.seed(42)
N = 50_000

fecha_inicio = pd.Timestamp("2018-01-01")
fecha_fin = pd.Timestamp("2025-12-31")

ramos = {
    "Auto": {"alpha": 1.8, "scale": 20_000},
    "Daños": {"alpha": 1.5, "scale": 50_000},
    "Gastos Médicos": {"alpha": 2.2, "scale": 30_000},
    "Vida": {"alpha": 1.3, "scale": 100_000}
}

clientes = [f"C{str(i).zfill(6)}" for i in range(1, 8001)]

# -----------------------------
# Generación de datos
# -----------------------------

# Fechas de ocurrencia
dias = (fecha_fin - fecha_inicio).days
fechas = fecha_inicio + pd.to_timedelta(
    np.random.randint(0, dias, size=N), unit="D"
)

# Ramos
ramo_asignado = np.random.choice(list(ramos.keys()), size=N, p=[0.4, 0.25, 0.25, 0.10])

# Clientes
cliente_asignado = np.random.choice(clientes, size=N)

# Montos (Pareto por ramo)
montos = np.zeros(N)

for ramo, params in ramos.items():
    idx = ramo_asignado == ramo
    alpha = params["alpha"]
    scale = params["scale"]

    montos[idx] = (
        np.random.pareto(alpha, size=idx.sum()) + 1
    ) * scale

# -----------------------------
# DataFrame final
# -----------------------------
reclamaciones = pd.DataFrame({
    "fecha_ocurrencia": fechas,
    "ramo": ramo_asignado,
    "cliente_id": cliente_asignado,
    "monto_reclamacion": montos.round(2)
})

# Ordenar por fecha
reclamaciones.sort_values("fecha_ocurrencia", inplace=True)

# -----------------------------
# Vista rápida
# -----------------------------
print(reclamaciones.head())
print("\nResumen por ramo:")
print(reclamaciones.groupby("ramo")["monto_reclamacion"].describe())

# Guardar a CSV
reclamaciones.to_csv("reclamaciones_simuladas.csv", index=False)


      fecha_ocurrencia            ramo cliente_id  monto_reclamacion
46775       2018-01-01            Auto    C003874           25690.30
36770       2018-01-01            Auto    C000756           24195.03
37606       2018-01-01            Vida    C004535          150428.50
17583       2018-01-01  Gastos Médicos    C002407           41080.55
27970       2018-01-01           Daños    C005122           74333.70

Resumen por ramo:
                  count           mean           std        min         25%  \
ramo                                                                          
Auto            19907.0   44321.788011  7.367409e+04   20000.31   23415.315   
Daños           12533.0  159653.589503  1.036078e+06   50001.22   60762.180   
Gastos Médicos  12663.0   55528.544020  5.319319e+04   30000.09   34266.520   
Vida             4897.0  434030.507464  2.639585e+06  100002.02  124586.870   

                      50%        75%           max  
ramo                                   